# Veritas Demo

In [7]:
%pip install -q "transformers>=4.46,<5" sentence-transformers rank_bm25 sentencepiece protobuf scikit-learn pandas pyarrow joblib torch youtube-transcript-api requests "gradio>=5,<6"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os, shutil, json
HERE = os.getcwd()
DRIVE_ROOT = os.path.join(HERE, "veritas")
NEED = ["claim_detector_v2", "retrieval", "verdict_model_v3"]
assert os.path.isdir(DRIVE_ROOT), f"Missing folder: {DRIVE_ROOT}. Put claim_detector_v2, retrieval and verdict_model_v3 inside a folder named 'veritas' next to this notebook."
for n in NEED:
    inner = os.path.join(DRIVE_ROOT, n, n)
    if os.path.isdir(inner):
        for f in os.listdir(inner): shutil.move(os.path.join(inner, f), os.path.join(DRIVE_ROOT, n, f))
        os.rmdir(inner)
import urllib.request
try: urllib.request.urlopen("https://pubmed.ncbi.nlm.nih.gov", timeout=8); print("internet: on")
except Exception: print("internet: OFF (live PubMed search will fall back to the built-in pool)")
import torch; print("device:", "cuda (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "cpu (works, but slower)")
print("veritas folder:", DRIVE_ROOT, "|", sorted(os.listdir(DRIVE_ROOT)))

internet: on
device: cpu (works, but slower)
veritas folder: c:\Users\JasonR\OneDrive\Desktop\Veritas - Temp\final_demo\final demo\veritas | []


In [9]:
import os, shutil
DRIVE_ROOT = r"c:\Users\JasonR\OneDrive\Desktop\Veritas - Temp"
for n in ["claim_detector_v2", "retrieval", "verdict_model_v3"]:
    assert os.path.isdir(os.path.join(DRIVE_ROOT, n)), f"missing {n} in {DRIVE_ROOT}"
    inner = os.path.join(DRIVE_ROOT, n, n)
    if os.path.isdir(inner):
        for f in os.listdir(inner): shutil.move(os.path.join(inner, f), os.path.join(DRIVE_ROOT, n, f))
        os.rmdir(inner)
    print(n, sorted(os.listdir(os.path.join(DRIVE_ROOT, n)))[:4])

claim_detector_v2 ['added_tokens.json', 'config.json', 'extract_config.json', 'model.safetensors']
retrieval ['aggregator.joblib', 'aggregator_prev.joblib', 'corpus.parquet', 'corpus_emb.npy']
verdict_model_v3 ['added_tokens.json', 'config.json', 'model.safetensors', 'special_tokens_map.json']


In [10]:
import os; print(os.getcwd()); print(os.listdir(os.getcwd()))

c:\Users\JasonR\OneDrive\Desktop\Veritas - Temp\final_demo\final demo
['README.md', 'veritas', 'veritas_demo.ipynb']


In [11]:
import os, re, json, html as H, collections, joblib, numpy as np, pandas as pd, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
DRIVE = DRIVE_ROOT
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
MAX_CLAIMS = 30

cfg = json.load(open(f"{DRIVE}/retrieval/retrieval_config.json"))
VERDICT_DIR = cfg.get("verdict_model", "verdict_model_v2")
GATE = float(cfg.get("relevance_gate", 0.15))
WEAK = 0.6
EMB_MODEL, QUERY_PREFIX, TOPK, NCAND, RERANK_MODEL = cfg["emb_model"], cfg["query_prefix"], cfg["topk"], cfg["ncand"], cfg["rerank_model"]
corpus = pd.read_parquet(f"{DRIVE}/retrieval/corpus.parquet"); D = np.load(f"{DRIVE}/retrieval/corpus_emb.npy")
agg = joblib.load(f"{DRIVE}/retrieval/aggregator.joblib")
assert len(corpus) == len(D), "corpus and embeddings are out of sync"

half = lambda m: m.half() if DEV == "cuda" else m
vtok = AutoTokenizer.from_pretrained(f"{DRIVE}/{VERDICT_DIR}")
vmodel = half(AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/{VERDICT_DIR}").to(DEV).eval())
ctok = AutoTokenizer.from_pretrained(f"{DRIVE}/claim_detector_v2")
cmodel = half(AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/claim_detector_v2").to(DEV).eval())
CLAIM_THR = json.load(open(f"{DRIVE}/claim_detector_v2/extract_config.json"))["threshold"]
st = SentenceTransformer(EMB_MODEL, device=DEV); st.max_seq_length = 256
rr = CrossEncoder(RERANK_MODEL, max_length=384, device=DEV)
if DEV == "cuda": rr.model.half()

tok_re = re.compile(r"[a-z0-9]+")
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())
toks = lambda t: [w for w in tok_re.findall(t.lower()) if w not in STOP]
bm25 = BM25Okapi([toks(t) for t in corpus.search_text])
print("loaded | device:", DEV, "| pool:", len(corpus), "| claim threshold:", CLAIM_THR, "| verdict model:", VERDICT_DIR, "| gate:", GATE)

C:\Users\JasonR\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\JasonR\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\JasonR\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\JasonR\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JasonR\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (11 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

loaded | device: cpu | pool: 5505 | claim threshold: 0.9 | verdict model: verdict_model_v3 | gate: 0.1


In [12]:
def rank_bm25_q(qs, n=100): return [np.argsort(-bm25.get_scores(toks(q)))[:n] for q in qs]
def rank_dense_q(qs, n=100):
    Q = st.encode([QUERY_PREFIX + q for q in qs], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    return [np.argsort(-s)[:n] for s in Q @ D.T]
def fuse(r1, r2, n=100, k=60):
    out = []
    for a, b in zip(r1, r2):
        sc = collections.defaultdict(float)
        for r in (a, b):
            for rank, i in enumerate(r): sc[i] += 1.0 / (k + rank + 1)
        out.append(np.array(sorted(sc, key=sc.get, reverse=True)[:n]))
    return out

def rerank(texts, rankings, ncand=NCAND, bs=64):
    pairs = [(t, corpus.search_text.iat[di]) for t, r in zip(texts, rankings) for di in r[:ncand]]
    sc = np.asarray(rr.predict(pairs, batch_size=bs, show_progress_bar=False), dtype="float64").reshape(len(texts), -1)
    if sc.min() < 0 or sc.max() > 1: sc = 1 / (1 + np.exp(-sc))
    nr, ns = [], []
    for r, s in zip(rankings, sc):
        o = np.argsort(-s); nr.append(r[:ncand][o]); ns.append(s[o])
    return nr, np.array(ns)

@torch.no_grad()
def verdict_probs(pairs, bs=32, max_len=384):
    out = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i + bs]
        enc = vtok([c for c, _ in b], [d for _, d in b], truncation="only_second", max_length=max_len, padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(vmodel(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def make_feats(P, S):
    S = S[:, :P.shape[1]]; w = S / (S.sum(1, keepdims=True) + 1e-9); wp = (w[:, :, None] * P).sum(1)
    return np.column_stack([P[:, :, 0].max(1), P[:, :, 1].max(1), P[:, :, 0].mean(1), P[:, :, 1].mean(1),
                            P[:, 0, 0], P[:, 0, 1], P[:, 0, 2], P[:, :, 2].mean(1), wp, S[:, 0], S[:, 0] - S[:, 1], S.mean(1)])

def check_claims(texts, show=3):
    rk, S = rerank(texts, fuse(rank_bm25_q(texts), rank_dense_q(texts)))
    pairs = [(t, corpus.text.iat[di]) for t, r in zip(texts, rk) for di in r[:TOPK]]
    P = verdict_probs(pairs).reshape(len(texts), TOPK, 3)
    A = agg.predict_proba(make_feats(P, S)); order = list(agg.classes_); out = []
    for i, t in enumerate(texts):
        probs = {LABELS[k]: float(A[i][order.index(k)]) if k in order else 0.0 for k in range(3)}
        gated = bool(S[i, 0] < GATE)
        verdict = "NEI" if gated else max(probs, key=probs.get)
        ev = [dict(title=corpus.title.iat[di] or "(untitled)", rerank=float(S[i, j]), passage=LABELS[int(P[i, j].argmax())],
                   p=float(P[i, j].max()), excerpt=corpus.text.iat[di][:420]) for j, di in enumerate(rk[i][:show])]
        out.append(dict(claim=t, verdict=verdict, confidence=None if gated else probs[verdict], gated=gated, weak=bool((not gated) and verdict != "NEI" and S[i, 0] < WEAK), probs=probs, evidence=ev))
    return out

@torch.no_grad()
def claim_probs(sents, bs=64):
    out = []
    for i in range(0, len(sents), bs):
        b = ctok(sents[i:i + bs], truncation=True, max_length=128, padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(cmodel(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def split_transcript(text, max_words=40, window=25):
    text = re.sub(r"\[?\(?\b\d{1,2}:\d{2}(?::\d{2})?\b\]?\)?", " ", text)
    text = re.sub(r"\s+", " ", text).strip(); out = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        w = s.split()
        if len(w) <= max_words:
            if len(w) >= 4: out.append(s)
        else:
            out += [" ".join(w[i:i + window]) for i in range(0, len(w), window) if len(w[i:i + window]) >= 4]
    return out

In [13]:
UNC = 0.5
import requests, time, xml.etree.ElementTree as ET

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
USE_LIVE = True
N_FILTERED, N_PLAIN = 8, 14
QUALITY = '(meta-analysis[pt] OR systematic review[pt] OR randomized controlled trial[pt] OR review[pt])'
GENERIC = set("""a an and are as at be been being by for from has have had in is it its of on or that the this to was were will with
you your yours we our they their he she his her i me my not no don doesn didn isn aren won do does did done can could would should may might must
more most less least very much many some any all each every than then so such just also only even ever never always
cure cures cured cause causes caused improve improves improved reduce reduces reduced increase increases increased decrease decreases
lead leads help helps helped good bad best better worse necessary important need needs needed give gives gave make makes made
get gets got take takes took people person things thing way ways really actually literally basically honestly truly
if when while because since though although how what which who whom whose why where there here
throughout within around about into over under through eating eat eats using use uses daily every day days per among between""".split())
_pm_cache, _q_cache, _last_call = {}, {}, [0.0]
LIVE_STATUS = ""

def _get(path, params, tries=3):
    for t in range(tries):
        wait = 0.36 - (time.time() - _last_call[0])
        if wait > 0: time.sleep(wait)
        _last_call[0] = time.time()
        try:
            r = requests.get(f"{EUTILS}/{path}", params={**params, "tool": "veritas-hackathon"}, timeout=25)
            if r.status_code == 200: return r
            if r.status_code in (429, 500, 502, 503): time.sleep(1.5 * (t + 1))
            else: break
        except requests.RequestException:
            time.sleep(1.0 * (t + 1))
    return None

def content_words(claim, k=4):
    ws = [w for w in re.findall(r"[a-z][a-z0-9\-]+", claim.lower()) if len(w) >= 3 and w not in GENERIC]
    top = max(bm25.idf.values()) * 1.2
    ws = sorted(dict.fromkeys(ws), key=lambda w: -bm25.idf.get(w, top))[:k]
    return ws

def pubmed_search(term, retmax):
    r = _get("esearch.fcgi", dict(db="pubmed", term=f"({term}) AND hasabstract[text]", retmax=retmax, sort="relevance", retmode="json"))
    if r is None: return None
    try: return r.json()["esearchresult"]["idlist"]
    except Exception: return None

def pubmed_fetch(pmids):
    docs, need = [], [p for p in pmids if p not in _pm_cache]
    if need:
        r = _get("efetch.fcgi", dict(db="pubmed", id=",".join(need), retmode="xml"))
        if r is None: return None
        try: root = ET.fromstring(r.content)
        except ET.ParseError: return None
        for art in root.findall(".//PubmedArticle"):
            pid = (art.findtext(".//MedlineCitation/PMID") or "").strip()
            title = "".join(art.find(".//ArticleTitle").itertext()).strip() if art.find(".//ArticleTitle") is not None else ""
            parts = []
            for a in art.findall(".//Abstract/AbstractText"):
                t = "".join(a.itertext()).strip()
                if t: parts.append((a.get("Label") + ": " if a.get("Label") else "") + t)
            year = art.findtext(".//JournalIssue/PubDate/Year") or (art.findtext(".//JournalIssue/PubDate/MedlineDate") or "")[:4]
            if pid and parts: _pm_cache[pid] = dict(title=title or "(untitled)", text=" ".join(parts), pmid=pid, year=year, src="pubmed")
    return [_pm_cache[p] for p in pmids if p in _pm_cache]

def live_docs(claim):
    ws = content_words(claim)
    if not ws: return []
    key = " ".join(ws)
    if key in _q_cache: return _q_cache[key]
    ids = []
    def add(term, n):
        got = pubmed_search(term, n)
        if got is None: return False
        ids.extend(i for i in got if i not in ids); return True
    ok = add(f"{' AND '.join(ws)} AND {QUALITY}", N_FILTERED)
    ok = add(' AND '.join(ws), N_PLAIN) and ok
    for k in range(len(ws) - 1, 1, -1):
        if len(ids) >= 12: break
        ok = add(' AND '.join(ws[:k]), N_PLAIN) and ok
    if not ids: return None if not ok else []
    docs = pubmed_fetch(ids)
    if docs is None: return None
    for d in docs: d["search_text"] = d["title"] + ". " + d["text"]
    _q_cache[key] = docs
    return docs

def check_claims_live(texts, show=3):
    global LIVE_STATUS
    pool_rk = fuse(rank_bm25_q(texts), rank_dense_q(texts))
    docs_per, n_live, failed = [], 0, 0
    for t, r in zip(texts, pool_rk):
        docs = [dict(title=corpus.title.iat[di] or "(untitled)", text=corpus.text.iat[di], search_text=corpus.search_text.iat[di], pmid=None, year="", src="pool") for di in r[:NCAND]]
        if USE_LIVE:
            lv = live_docs(t)
            if lv is None: failed += 1
            else:
                seen = {re.sub(r"\W+", "", d["title"].lower()) for d in docs}
                new = [d for d in lv if re.sub(r"\W+", "", d["title"].lower()) not in seen]; n_live += len(new); docs += new
        docs_per.append(docs)
    pairs = [(t, d["search_text"]) for t, docs in zip(texts, docs_per) for d in docs]
    sc = np.asarray(rr.predict(pairs, batch_size=64, show_progress_bar=False), dtype="float64")
    if sc.size and (sc.min() < 0 or sc.max() > 1): sc = 1 / (1 + np.exp(-sc))
    top, S, o = [], [], 0
    for docs in docs_per:
        s = sc[o:o + len(docs)]; o += len(docs); order = np.argsort(-s)
        top.append([docs[j] for j in order[:max(TOPK, show)]]); S.append(s[order][:max(TOPK, show)])
    S = np.array(S)
    P = verdict_probs([(t, d["text"]) for t, tp in zip(texts, top) for d in tp[:TOPK]]).reshape(len(texts), TOPK, 3)
    A = agg.predict_proba(make_feats(P, S)); order_c = list(agg.classes_); out = []
    for i, t in enumerate(texts):
        probs = {LABELS[k]: float(A[i][order_c.index(k)]) if k in order_c else 0.0 for k in range(3)}
        gated = bool(S[i, 0] < GATE)
        verdict = "NEI" if gated else max(probs, key=probs.get)
        lean = verdict
        if verdict in ("SUPPORTS", "REFUTES") and probs[verdict] < UNC: verdict = "UNCERTAIN"
        ev = [dict(title=d["title"], rerank=float(S[i, j]), passage=LABELS[int(P[i, j].argmax())], p=float(P[i, j].max()),
                   excerpt=d["text"][:420], pmid=d["pmid"], year=d["year"], src=d["src"]) for j, d in enumerate(top[i][:show])]
        out.append(dict(claim=t, verdict=verdict, confidence=None if gated else probs[lean], lean=lean, gated=gated,
                        weak=bool((not gated) and verdict in ("SUPPORTS", "REFUTES") and S[i, 0] < WEAK), probs=probs, evidence=ev))
    LIVE_STATUS = ("Evidence: built-in pool + live PubMed search" if USE_LIVE and failed < len(texts) else
                   "Evidence: built-in pool only (PubMed could not be reached)" if USE_LIVE else "Evidence: built-in pool only")
    if USE_LIVE and 0 < failed < len(texts): LIVE_STATUS += f" ({failed} claim(s) fell back to pool only)"
    return out

check_claims_pool = check_claims
check_claims = check_claims_live

def ev_tag(e):
    if e.get("pmid"):
        return (f" <a href='https://pubmed.ncbi.nlm.nih.gov/{e['pmid']}/' target='_blank' style='color:#2563eb !important;font-size:12px'>PubMed {e['year']} ↗</a>")
    return " <span style='color:#6b7280 !important;font-size:12px'>(built-in pool)</span>"

In [14]:
V = {"SUPPORTS": ("Supported by evidence", "#10b981", "#ecfdf5"), "REFUTES": ("Contradicted by evidence", "#ef4444", "#fef2f2"),
     "NEI": ("Not enough evidence", "#6b7280", "#f3f4f6"), "UNCERTAIN": ("Uncertain: evidence is mixed", "#d97706", "#fffbeb")}
conf_label = lambda c: (lambda c: "high" if c >= 0.70 else "medium" if c >= 0.50 else "low")(round(c, 2))
INK = "color:#111827 !important;"
WRAP = lambda h: "<div style='background:#ffffff;padding:12px;border-radius:8px;color:#111827 !important'>" + h + "</div>"

def card(r, i):
    name, border, bg = V[r["verdict"]]
    if r["gated"]:   conf = "no relevant abstract found"
    elif r["verdict"] == "UNCERTAIN": conf = f"no clear direction (leans {r['lean'].lower()}, {r['confidence']:.0%})"
    elif r["weak"]:  conf = f"confidence low ({r['confidence']:.0%}) · only loosely related abstracts were found, so treat this with caution"
    else:            conf = f"confidence {conf_label(r['confidence'])} ({r['confidence']:.0%})"
    ev = "".join(
        f"<details style='margin:4px 0'><summary style='cursor:pointer;{INK}font-size:13px'>"
        f"<b style='{INK}'>{H.escape(e['title'][:110])}</b>{ev_tag(e)} &nbsp;<span style='color:#4b5563 !important'>reads as {e['passage'].lower()} ({e['p']:.0%}) · relevance {e['rerank']:.2f}</span></summary>"
        f"<div style='font-size:12px;color:#374151 !important;padding:4px 0 4px 12px'>{H.escape(e['excerpt'])}…</div></details>" for e in r["evidence"])
    return (f"<div style='border-left:6px solid {border};background:{bg};padding:10px 14px;margin:10px 0;border-radius:6px;{INK}'>"
            f"<div style='font-size:12px;color:#4b5563 !important'>CLAIM {i}</div>"
            f"<div style='font-size:15px;margin:2px 0 6px 0;{INK}'>{H.escape(r['claim'])}</div>"
            f"<div style='font-size:14px;{INK}'><b style='color:{border} !important'>{name}</b> &nbsp;·&nbsp; {conf}</div>"
            f"<div style='margin-top:6px;{INK}'>{ev}</div></div>")

def analyze(transcript, thr=CLAIM_THR):
    sents = split_transcript(transcript or "")
    if not sents: return WRAP("<i>Paste a transcript first.</i>"), ""
    p = claim_probs(sents); flagged = [i for i, x in enumerate(p) if x >= thr]
    note = ""
    if len(flagged) > MAX_CLAIMS:
        flagged = sorted(sorted(flagged, key=lambda i: -p[i])[:MAX_CLAIMS]); note = f" (showing the {MAX_CLAIMS} most claim-like)"
    res = check_claims([sents[i] for i in flagged]) if flagged else []
    by = {i: r for i, r in zip(flagged, res)}
    cnt = collections.Counter(r["verdict"] for r in res)
    summary = (f"<div style='{INK}font-size:14px;margin-bottom:6px'><b>{len(sents)}</b> sentences · <b>{len(flagged)}</b> checkable claims{note} · "
               f"<span style='color:#10b981 !important'><b>{cnt['SUPPORTS']}</b> supported</span> · <span style='color:#ef4444 !important'><b>{cnt['REFUTES']}</b> contradicted</span> · "
               f"<span style='color:#d97706 !important'><b>{cnt['UNCERTAIN']}</b> uncertain</span> · <span style='color:#6b7280 !important'><b>{cnt['NEI']}</b> not enough evidence</span><br><span style='color:#6b7280 !important;font-size:12px'>{LIVE_STATUS}</span></div>")
    cards = "".join(card(r, n + 1) for n, r in enumerate(res)) or "<i>No checkable claims found. Try lowering the claim threshold.</i>"
    spans = []
    for i, s in enumerate(sents):
        if i in by:
            _, border, bg = V[by[i]["verdict"]]
            spans.append(f"<span style='background:{bg};border-bottom:3px solid {border};padding:1px 3px;{INK}'>{H.escape(s)}</span>")
        else:
            spans.append(f"<span style='color:#6b7280 !important'>{H.escape(s)}</span>")
    view = "<div style='line-height:1.9;font-size:14px'>" + " ".join(spans) + "</div>"
    return WRAP(summary + cards), WRAP(view)

In [15]:
def fetch_captions(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", url or "")
    if not m: return "", "Could not find a video id in that link."
    vid = m.group(1)
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        try:    segs = [s.text for s in YouTubeTranscriptApi().fetch(vid)]
        except AttributeError: segs = [s["text"] for s in YouTubeTranscriptApi.get_transcript(vid)]
        return " ".join(segs), f"Fetched {len(segs)} caption lines."
    except Exception as e:
        return "", f"Could not fetch captions ({type(e).__name__}). Paste the transcript instead."

In [16]:
DEMO1 = '''Hey everyone, welcome back to the channel. Today I want to talk about sleep and supplements. So first, I take magnesium glycinate every night and honestly it cures my insomnia. Studies show that vitamin D supplementation reduces the risk of respiratory infections. I personally think cold plunges are amazing. Masks do not reduce the spread of COVID-19, that's just a fact. Intermittent fasting is the best diet in my opinion. Drinking lemon water every morning cures cancer. Make sure you like and subscribe. The Eiffel Tower is 330 meters tall. Statins cause more harm than good for most people. Anyway, let me know what you think in the comments.'''
DEMO2 = '''Okay so, um, last month my doctor told me my cholesterol was high. A lot of people say eggs raise your cholesterol but the research really doesn't back that up. I swapped sugar for honey and my energy is way better. Drinking eight glasses of water a day is necessary for good health. Seed oils are poison, they cause inflammation throughout the body. Honestly this smoothie recipe is so good. Sunscreen causes more cancer than it prevents. If you walk ten thousand steps a day you will live longer. Alright, that's it for today, thanks for watching.'''
for name, T in [("DEMO1", DEMO1), ("DEMO2", DEMO2)]:
    sents = split_transcript(T); p = claim_probs(sents); fl = [i for i, x in enumerate(p) if x >= CLAIM_THR]
    print(f"\n{name}: {len(sents)} sentences, {len(fl)} claims")
    for i, r in zip(fl, check_claims([sents[i] for i in fl])):
        conf = "no relevant abstract" if r["gated"] else f"{r['confidence']:.2f}" + (" (weak)" if r["weak"] else "")
        print(f"  {r['verdict']:8s} {conf:>20s} | {r['claim'][:80]}")


DEMO1: 12 sentences, 6 claims
  NEI      no relevant abstract | So first, I take magnesium glycinate every night and honestly it cures my insomn
  SUPPORTS                 0.65 | Studies show that vitamin D supplementation reduces the risk of respiratory infe
  REFUTES                  0.56 | Masks do not reduce the spread of COVID-19, that's just a fact.
  REFUTES           0.62 (weak) | Drinking lemon water every morning cures cancer.
  NEI                      0.77 | The Eiffel Tower is 330 meters tall.
  UNCERTAIN                 0.44 | Statins cause more harm than good for most people.

DEMO2: 9 sentences, 6 claims
  NEI      no relevant abstract | A lot of people say eggs raise your cholesterol but the research really doesn't 
  NEI      no relevant abstract | I swapped sugar for honey and my energy is way better.
  UNCERTAIN                 0.45 | Drinking eight glasses of water a day is necessary for good health.
  UNCERTAIN                 0.49 | Seed oils are poison, they ca

In [17]:
CHECK = [("TRUE", "Creatine gives your muscles more energy."), ("TRUE", "If you don't drink water, you will die."), ("TRUE", "If you do not eat food, you will die."),
         ("TRUE", "Breathing oxygen is necessary for human survival."), ("TRUE", "Sleep is important for good health."),
         ("FALSE", "Drinking lemon water cures cancer."), ("FALSE", "Magnesium supplements cure insomnia."),
         ("FALSE", "Eating carrots gives you night vision."), ("?", "Regular exercise improves heart health."),
         ("?", "Vitamin D supplementation reduces the risk of respiratory infections.")]
res = check_claims([c for _, c in CHECK])
rows = []
for (truth, c), r in zip(CHECK, res):
    shown = "no relevant abstract" if r["gated"] else (f"{r['verdict']} {r['confidence']:.2f}" + (" (weak-evidence warning)" if r["weak"] else ""))
    rows.append(dict(truth=truth, claim=c[:52], demo_shows=shown, top_relevance=round(r["evidence"][0]["rerank"], 2), top_abstract=r["evidence"][0]["title"][:48], src=r["evidence"][0]["src"]))
print(pd.DataFrame(rows).to_string(index=False))
def confident(r): return (not r["gated"]) and (not r["weak"])
bad_true = [c for (t, c), r in zip(CHECK, res) if t == "TRUE" and r["verdict"] == "REFUTES" and confident(r)]
bad_false = [c for (t, c), r in zip(CHECK, res) if t == "FALSE" and r["verdict"] == "SUPPORTS" and confident(r)]
print("\nobviously-true claims shown as a confident 'Contradicted':", len(bad_true), bad_true)
print("obviously-false claims shown as a confident 'Supported':", len(bad_false), bad_false)

truth                                                claim                           demo_shows  top_relevance                                     top_abstract    src
 TRUE             Creatine gives your muscles more energy.                       UNCERTAIN 0.45           0.71 Effects of creatine supplementation on muscle st pubmed
 TRUE              If you don't drink water, you will die.                         REFUTES 0.79           0.95 [Water intoxication--a little known complication pubmed
 TRUE                If you do not eat food, you will die.                         REFUTES 0.52           0.98 Dairy product consumption reduces cardiovascular pubmed
 TRUE    Breathing oxygen is necessary for human survival.                             NEI 0.63           0.13 Nitrite protects against morbidity and mortality   pool
 TRUE                  Sleep is important for good health.                        SUPPORTS 0.79           1.00  Sleep health: can we define it? Does it matter? pubme

In [20]:
import gradio as gr

LIMITS = '''**How it works.** A claim detector picks checkable sentences. Each claim is searched against PubMed abstracts and a 5,505-abstract pool, re-ranked, and judged by a fine-tuned DeBERTa verdict model.

** limits**
the biggest issue is that it lacks data. when theres a lack of data, it just pulls any relevant topic, so sometimes super obvious statements are missed. if the claim is "if you dont drink water, youll die," itll disagree or be unsure. 
this is due to the fact that theres no research on it. if someone really did a study that tested if people who didnt drink water lived, then it'd be able to answer well'''

with gr.Blocks(title="Veritas: health claim checker") as demo:
    gr.Markdown("# Veritas — health claim checker\nPaste a video transcript. Veritas finds the health claims and checks them against online research. Fair warning! the model is pretty slow, so you might have to wait a lil bit.")
    with gr.Row():
        with gr.Column(scale=1):
            tx = gr.Textbox(label="Transcript", lines=12, placeholder="Paste a transcript here…")
            with gr.Row():
                url = gr.Textbox(label="…or a YouTube link", scale=3); fetch = gr.Button("Fetch captions", scale=1)
            status = gr.Markdown("")
            thr = gr.Slider(0.3, 0.99, value=CLAIM_THR, step=0.01, label="Claim threshold (lower = flag more sentences)")
            go = gr.Button("Check claims", variant="primary")
            gr.Examples(examples=[[DEMO1], [DEMO2]], inputs=[tx], label="Invented example transcripts")
        with gr.Column(scale=1):
            out = gr.HTML(label="Results")
            with gr.Accordion("Transcript view", open=False): view = gr.HTML()
    with gr.Accordion("limitations", open=False): gr.Markdown(LIMITS)
    fetch.click(fetch_captions, [url], [tx, status])
    go.click(analyze, [tx, thr], [out, view])

demo.launch(share=True, debug=False)

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
